# 样本制作步骤（查询城市 & 目标城市）

本 Notebook 独立完成滑动窗口样本生成：
- 窗口 192，输入 168，预测 24，步长 6
- 查询城市：1991-2023 数据
- 目标城市：汉中，2024 数据（用于后续小模型）

## 1. 参数与路径
- `DATA_ROOT`：原始温度数据所在目录
- 输出将写入 `samples_w192_s6/query` 与 `samples_w192_s6/target`

In [1]:
from pathlib import Path
import numpy as np

# 滑动窗口配置
WINDOW = 192
INPUT_LEN = 168
STEP = 6

# 数据根目录（按需修改）
DATA_ROOT = Path('./shaanxi_cities_temperature')
OUT_ROOT = DATA_ROOT / f'samples_w{WINDOW}_s{STEP}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# 城市与年份标签
QUERY_CITIES = ['西安','郑州','太原','石家庄','济南','呼和浩特','银川','兰州','西宁','大同']
TARGET_CITIES = ['汉中']
QUERY_TAG = '1991-2023'
TARGET_TAG = '2024'

print('DATA_ROOT =', DATA_ROOT)
print('OUT_ROOT  =', OUT_ROOT)

DATA_ROOT = shaanxi_cities_temperature
OUT_ROOT  = shaanxi_cities_temperature\samples_w192_s6


## 2. 辅助函数
- `find_files`：在子目录中查找 `<city>_<tag>.npy` 与对应时间文件
- `make_samples`：按滑动窗口生成样本并保存

In [2]:
from typing import Tuple


def find_files(city: str, tag: str) -> Tuple[Path, Path]:
    data = sorted(DATA_ROOT.glob(f'**/{city}_{tag}.npy'))
    times = sorted(DATA_ROOT.glob(f'**/{city}_{tag}_times.npy'))
    return (data[0] if data else None, times[0] if times else None)


def make_samples(city: str, data_path: Path, time_path: Path, out_dir: Path):
    data = np.load(data_path)
    times = np.load(time_path)
    if len(data) != len(times):
        print(f"{city}: 数据与时间长度不一致，跳过")
        return
    if len(data) < WINDOW:
        print(f"{city}: 数据长度 {len(data)} < 窗口 {WINDOW}，跳过")
        return

    inputs, targets, starts = [], [], []
    for start in range(0, len(data) - WINDOW + 1, STEP):
        window = data[start:start+WINDOW]
        inputs.append(window[:INPUT_LEN])
        targets.append(window[INPUT_LEN:WINDOW])
        starts.append(times[start])

    inputs = np.stack(inputs)
    targets = np.stack(targets)
    starts = np.array(starts)

    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f"{city}_samples_w{WINDOW}_s{STEP}.npz"
    np.savez_compressed(out_file, input=inputs, target=targets, start_time=starts)
    print(f"{city}: 生成 {len(inputs)} 条样本 -> {out_file}")

## 3. 生成查询城市样本（1991-2023）

In [3]:
for city in QUERY_CITIES:
    d, t = find_files(city, QUERY_TAG)
    if not d or not t:
        print(f"{city}: 未找到 {city}_{QUERY_TAG}.npy 或时间文件，跳过")
        continue
    make_samples(city, d, t, OUT_ROOT / 'query')

西安: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\西安_samples_w192_s6.npz
郑州: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\郑州_samples_w192_s6.npz
太原: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\太原_samples_w192_s6.npz
石家庄: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\石家庄_samples_w192_s6.npz
济南: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\济南_samples_w192_s6.npz
呼和浩特: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\呼和浩特_samples_w192_s6.npz
银川: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\银川_samples_w192_s6.npz
兰州: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\兰州_samples_w192_s6.npz
西宁: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\西宁_samples_w192_s6.npz
大同: 生成 48181 条样本 -> shaanxi_cities_temperature\samples_w192_s6\query\大同_samples_w192_s6.npz


## 4. 生成目标城市样本（汉中，2024）

In [4]:
for city in TARGET_CITIES:
    d, t = find_files(city, TARGET_TAG)
    if not d or not t:
        print(f"{city}: 未找到 {city}_{TARGET_TAG}.npy 或时间文件，跳过")
        continue
    make_samples(city, d, t, OUT_ROOT / 'target')

汉中: 生成 1433 条样本 -> shaanxi_cities_temperature\samples_w192_s6\target\汉中_samples_w192_s6.npz


## 5. 查看输出

In [5]:
import glob
print('样本文件：')
for f in sorted(glob.glob(str((OUT_ROOT) / '**/*.npz'), recursive=True)):
    print(f)

样本文件：
shaanxi_cities_temperature\samples_w192_s6\query\兰州_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\呼和浩特_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\大同_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\太原_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\济南_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\石家庄_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\西宁_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\西安_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\郑州_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\query\银川_samples_w192_s6.npz
shaanxi_cities_temperature\samples_w192_s6\target\汉中_samples_w192_s6.npz
